# 03 - Baseline Restoration Methods
Simple non-deep-learning baselines (bicubic upsampling, Gaussian/median/bilateral filtering) to compare against NAFNet.

In [ ]:
import cv2, numpy as np, torch, matplotlib.pyplot as plt, os
from metrics import evaluate_batch  # from src/

In [ ]:
def bicubic_baseline(img):
    return torch.nn.functional.interpolate(img.unsqueeze(0), size=img.shape[-2:], mode='bicubic', align_corners=False)[0].clamp(0,1)

def median_filter_baseline(img_np, ksize=3):
    return cv2.medianBlur((img_np*255).astype(np.uint8), ksize).astype(np.float32)/255.0

def bilateral_filter_baseline(img_np, d=5, sigma_color=50, sigma_space=50):
    return cv2.bilateralFilter((img_np*255).astype(np.uint8), d, sigma_color, sigma_space).astype(np.float32)/255.0

In [ ]:
noisy = np.load(os.path.join(noisy_dir, sorted(os.listdir(noisy_dir))[0])).astype(np.float32)
clean = np.load(os.path.join(clean_dir, sorted(os.listdir(clean_dir))[0])).astype(np.float32)
if noisy.max() > 1.5: noisy /= 255.0
if clean.max() > 1.5: clean /= 255.0

median_out = median_filter_baseline(noisy)
bilateral_out = bilateral_filter_baseline(noisy)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes, [noisy, median_out, bilateral_out, clean],
                            ["Degraded", "Median Filter", "Bilateral Filter", "Clean GT"]):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def psnr_np(a, b):
    mse = np.mean((a-b)**2)
    return 10*np.log10(1.0/mse) if mse > 0 else float('inf')

print("Degraded  PSNR:", psnr_np(noisy, clean))
print("Median    PSNR:", psnr_np(median_out, clean))
print("Bilateral PSNR:", psnr_np(bilateral_out, clean))